In [1]:
!pip install -U transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/capstone/data"

TEST_FILE = f"{DATA_DIR}/test.jsonl"

MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1"

print(TEST_FILE)
print(MODEL_PATH)

/content/drive/MyDrive/Colab Notebooks/capstone/data/test.jsonl
/content/drive/MyDrive/Colab Notebooks/capstone/models/cuad-clause-extractor-v1


In [5]:
import os

print("Test file:", os.path.exists(TEST_FILE))
print("Model folder:", os.path.exists(MODEL_PATH))

print(os.listdir(MODEL_PATH))

Test file: True
Model folder: True
['checkpoint-82', 'checkpoint-164', 'checkpoint-246', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "test": TEST_FILE
    }
)

dataset

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['messages'],
        num_rows: 41
    })
})

In [7]:
dataset["test"][0]

{'messages': [{'role': 'system',
   'content': 'You are a legal contract analysis assistant. Extract important clauses and return JSON.'},
  {'role': 'user',
   'content': 'EXHIBIT 10.2\n\n                    SITE DEVELOPMENT AND HOSTING AGREEMENT\n\n                  This SITE DEVELOPMENT AND HOSTING AGREEMENT (the "Agreement") dated as of August 9, 1999 is made between Hanover Direct, Inc. ("HDI"), a New Jersey Corporation, located at 1500 Harbor Boulevard, Weehawken, NJ 07087, and The Deerskin Companies, Inc. (the "Company"), a Nevada corporation, located at 2500 Arrowhead Drive, Carson City, NV 89706. Each of the parties hereto shall be referred to as a "Party".\n\n                  In consideration of the mutual promises and covenants set forth below, HDI and the Company agree as follows:\n\n1. HDI\'s Responsibilities.\n\n                  1.1 HDI shall design, develop, implement, operate, maintain and manage, and enable the Company to establish a presence on the World Wide Web ("

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")

Tokenizer loaded


In [9]:
from transformers import AutoModelForCausalLM

base_model = "HuggingFaceTB/SmolLM2-360M-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

print("Base model loaded")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Base model loaded


In [11]:
!pip uninstall -y torchao
!pip install -U torchao peft transformers accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.5 MB/s eta 0:00:00


In [12]:
import peft
import torchao

print("peft:", peft.__version__)
print("torchao:", torchao.__version__)

peft: 0.19.1
torchao: 0.17.0


In [13]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,
    MODEL_PATH
)

model.eval()

print("Fine-tuned CUAD model loaded")

Fine-tuned CUAD model loaded


In [23]:
import torch

def generate_prediction(text):

    prompt = f"""
Extract these legal clauses from the contract:

- governing_law
- termination
- indemnification
- liability_cap
- exclusivity

Contract:
{text}

Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)


    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )


    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result

In [16]:
sample = dataset["test"][0]

print(sample.keys())
print(sample)

dict_keys(['messages'])
{'messages': [{'role': 'system', 'content': 'You are a legal contract analysis assistant. Extract important clauses and return JSON.'}, {'role': 'user', 'content': 'EXHIBIT 10.2\n\n                    SITE DEVELOPMENT AND HOSTING AGREEMENT\n\n                  This SITE DEVELOPMENT AND HOSTING AGREEMENT (the "Agreement") dated as of August 9, 1999 is made between Hanover Direct, Inc. ("HDI"), a New Jersey Corporation, located at 1500 Harbor Boulevard, Weehawken, NJ 07087, and The Deerskin Companies, Inc. (the "Company"), a Nevada corporation, located at 2500 Arrowhead Drive, Carson City, NV 89706. Each of the parties hereto shall be referred to as a "Party".\n\n                  In consideration of the mutual promises and covenants set forth below, HDI and the Company agree as follows:\n\n1. HDI\'s Responsibilities.\n\n                  1.1 HDI shall design, develop, implement, operate, maintain and manage, and enable the Company to establish a presence on the W

In [17]:
!pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ca911f3bd6b576a1e3825dca7514bc3ae8dfd0771ec75266846cddf7422d9ccc
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [18]:
import evaluate

rouge = evaluate.load("rouge")

In [20]:
def get_expected(sample):

    for msg in sample["messages"]:
        if msg["role"] == "assistant":
            return msg["content"]

    return ""

In [22]:
predictions = []
references = []

for sample in dataset["test"].select(range(2)):

    contract = ""

    for msg in sample["messages"]:
        if msg["role"] == "user":
            contract = msg["content"]

    prediction = generate_prediction(contract)

    reference = get_expected(sample)

    predictions.append(prediction)
    references.append(reference)


print("Completed predictions")

Completed predictions


In [24]:
results = rouge.compute(
    predictions=predictions,
    references=references
)

results

{'rouge1': np.float64(0.12986129155994408),
 'rouge2': np.float64(0.06738084990290408),
 'rougeL': np.float64(0.0898944528997612),
 'rougeLsum': np.float64(0.11813108628135169)}

In [25]:
print("Phase 5 Evaluation Completed")
print("----------------------------")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")
print(f"ROUGE-Lsum: {results['rougeLsum']:.4f}")

Phase 5 Evaluation Completed
----------------------------
ROUGE-1: 0.1299
ROUGE-2: 0.0674
ROUGE-L: 0.0899
ROUGE-Lsum: 0.1181
